# 数理変換タクティクス AI トレーニング
## ハイブリッドEmbedding版（数学特徴 + 学習特徴 + 継続学習 + リソース対応）

In [ ]:
!pip install tensorflowjs scikit-learn optuna -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random, os, math
from sklearn.model_selection import train_test_split

MAX_HAND    = 20
EMBED_DIM   = 16
NUM_ACTIONS = 7
MAX_CARD    = 150
PRIMES      = [2, 3, 5, 7, 11, 13]
MATH_DIM    = len(PRIMES) + 3
SCALAR_DIM  = 10
DIM_NAMES   = ['2の指数','3の指数','5の指数','7の指数','11の指数','13の指数','大素数フラグ','約数の個数','桁の和']

print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ================================================================
# ⚙️ 設定セル ── ここだけ変更すればOK！
# ================================================================

# ─── データ生成 ──────────────────────────────────────────────────
NUM_EPISODES = 30000     # 生成ゲーム数（多い→精度UP・時間もかかる）

# ─── ルール多様性 ────────────────────────────────────────────────
# 行をコメントアウトすると除外、追加で増やせる
# ※ initHandCount=5, winScore=200, maxTurns=20, resourceLogBase=2 で固定
RULE_CONFIGS = [
    # resourceInitial=5,  最低値なし
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':5,   'resourceMin':0,  'resourceLogBase':2},
    # resourceInitial=10, 最低値なし
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':10,  'resourceMin':0,  'resourceLogBase':2},
    # resourceInitial=10, 最低値=5
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':10,  'resourceMin':5,  'resourceLogBase':2},
    # resourceInitial=15, 最低値なし
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':15,  'resourceMin':0,  'resourceLogBase':2},
    # resourceInitial=15, 最低値=10
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':15,  'resourceMin':10, 'resourceLogBase':2},
    # リソースなし
    {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':None,'resourceMin':0,  'resourceLogBase':2},
]

# ─── 学習設定 ─────────────────────────────────────────────────────
EPOCHS              = 60     # 最大エポック数（EarlyStoppingで早めに止まることも）
BATCH_SIZE          = 256    # バッチサイズ（大きい→速い、小さい→丁寧）
LEARNING_RATE       = 0.001  # 学習率（大きい→速いが不安定、小さい→慎重）
EARLY_STOP_PATIENCE = 6      # 改善なし N エポックで学習を停止
LR_REDUCE_PATIENCE  = 3      # 改善なし N エポックで学習率を半分に下げる

# ─── 数学特徴 重要度テスト ──────────────────────────────────────
IMPORTANCE_INTERVAL = 0      # 何エポックごとに重要度テストを実行するか
                              # 0 = 学習後に1回だけ / 例: 10 = 10エポックごと

# ─── Optuna ハイパーパラメータ自動探索 ──────────────────────────
USE_OPTUNA       = True  # True にすると学習前に自動探索を実行
OPTUNA_TRIALS    = 20    # 試行回数（多い→より良いパラメータ、時間もかかる）
OPTUNA_EPOCHS    = 15    # 1試行あたりのエポック数（本番の1/4程度が目安）

print('設定読み込み完了！')
print(f'  ゲーム数: {NUM_EPISODES}  ルール数: {len(RULE_CONFIGS)}')
print(f'  エポック: {EPOCHS}  バッチ: {BATCH_SIZE}  学習率: {LEARNING_RATE}')
print(f'  重要度テスト: {"学習後のみ" if IMPORTANCE_INTERVAL == 0 else f"{IMPORTANCE_INTERVAL}エポックごと"}')
print(f'  Optuna: {"有効 (" + str(OPTUNA_TRIALS) + "試行)" if USE_OPTUNA else "無効"}')

## Google Drive 連携

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR  = '/content/drive/MyDrive/math_tactics_ai'
SAVE_PATH = os.path.join(SAVE_DIR, 'ai_best.keras')
DATA_PATH = os.path.join(SAVE_DIR, 'train_data_v3.npz')
os.makedirs(SAVE_DIR, exist_ok=True)

print('モデル保存先: ' + SAVE_PATH)
print('データ保存先: ' + DATA_PATH)
if os.path.exists(SAVE_PATH): print('保存済みモデルあり')
if os.path.exists(DATA_PATH): print('保存済みデータあり')

## 数学特徴テーブル

In [ ]:
def _card_features_full(n):
    """カード値 n の全数学特徴を計算（14次元）"""
    if n == 0: return [0.0] * 14
    orig = n
    # ── 1〜6: 素因数の指数 (2,3,5,7,11,13) ──
    feats = []
    tmp = n
    for p in PRIMES:
        c = 0
        while tmp % p == 0: tmp //= p; c += 1
        feats.append(min(c, 4) / 4.0)
    # ── 7: 大素数フラグ (13より大きい素因数あり) ──
    feats.append(1.0 if tmp > 1 else 0.0)
    # ── 8: 約数の個数 ──
    feats.append(min(sum(1 for d in range(1, orig+1) if orig % d == 0), 20) / 20.0)
    # ── 9: 桁の和 ──
    feats.append(min(sum(int(d) for d in str(orig)), 30) / 30.0)
    # ── 10: 素数フラグ ──
    is_p = orig > 1 and all(orig % d != 0 for d in range(2, int(orig**0.5)+1))
    feats.append(1.0 if is_p else 0.0)
    # ── 11: 最小素因数（正規化） ──
    min_pf = orig
    for d in range(2, int(orig**0.5)+1):
        if orig % d == 0: min_pf = d; break
    feats.append(min(min_pf, 150) / 150.0)
    # ── 12: 最大素因数（正規化） ──
    max_pf, tmp2 = 1, orig
    for d in range(2, int(orig**0.5)+1):
        while tmp2 % d == 0: tmp2 //= d; max_pf = d
    if tmp2 > 1: max_pf = tmp2
    feats.append(min(max_pf, 150) / 150.0)
    # ── 13: 素因数の個数Ω（重複あり） ──
    omega, tmp3 = 0, orig
    for p in list(PRIMES) + list(range(17, int(orig**0.5)+1)):
        while tmp3 % p == 0: tmp3 //= p; omega += 1
    if tmp3 > 1: omega += 1
    feats.append(min(omega, 8) / 8.0)
    # ── 14: 約数の和 σ(n)（正規化） ──
    sigma = sum(d for d in range(1, orig+1) if orig % d == 0)
    feats.append(min(sigma, orig * 6) / max(orig * 6, 1))
    return feats

# 全特徴テーブル（151枚 × 14次元）
ALL_DIM_NAMES = [
    '2の指数','3の指数','5の指数','7の指数','11の指数','13の指数',  # 0-5
    '大素数フラグ','約数の個数','桁の和',                           # 6-8（ここまで従来）
    '素数フラグ','最小素因数','最大素因数','素因数個数Ω','約数の和σ', # 9-13（新規）
]
ALL_FACTOR_TABLE = np.array([_card_features_full(i) for i in range(MAX_CARD+1)], dtype=np.float32)

# デフォルト: 従来の9次元（Optunaが最適化するまでの初期値）
MATH_FEATURE_MASK = list(range(9))
FACTOR_TABLE = ALL_FACTOR_TABLE[:, MATH_FEATURE_MASK]
MATH_DIM     = len(MATH_FEATURE_MASK)
DIM_NAMES    = [ALL_DIM_NAMES[i] for i in MATH_FEATURE_MASK]

print(f'全特徴テーブル: {ALL_FACTOR_TABLE.shape}')
print(f'使用中: {MATH_DIM}次元 → {DIM_NAMES}')

## ゲームシミュレーター（リソース対応）

In [ ]:
def gcd(a,b):
    while b: a,b=b,a%b
    return a
def digit_sum(n): return sum(int(d) for d in str(n))

class MathCardGame:
    def __init__(self, config=None):
        self.config = config or {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':10,'resourceMin':0,'resourceLogBase':2}
    def reset(self):
        cfg=self.config
        self.hands={'P1':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],'P2':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])]}
        self.scores={'P1':0,'P2':0}; self.turn_count=1; self.current_player='P1'
        self.done=False; self.winner=None
        ri = cfg.get('resourceInitial', None)
        self.resources = {'P1': ri, 'P2': ri}
    def opponent(self,pid): return 'P2' if pid=='P1' else 'P1'
    def _res_delta(self, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return 0.0
        rm = self.config.get('resourceMin', 0) or 0
        effective_before = max(sum(hb), rm)
        diff = sum(ha) - effective_before
        if diff == 0: return 0.0
        base = max(self.config.get('resourceLogBase', 2), 1.001)
        return (1 if diff>0 else -1) * math.log(abs(diff)+1) / math.log(base)
    def _can_afford(self, pid, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return True
        return (self.resources.get(pid, ri) + self._res_delta(hb, ha)) >= 0
    def _use_resource(self, pid, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return
        self.resources[pid] = self.resources.get(pid, ri) + self._res_delta(hb, ha)
    def _res_ratio(self, pid):
        ri=self.config.get('resourceInitial', None)
        rm=self.config.get('resourceMin', 0) or 0
        if ri is None: return 1.0
        range_ = ri - rm
        if range_ <= 0: return 1.0
        current = self.resources.get(pid, ri)
        return min(max((current - rm) / range_, 0), 2.0)
    def get_inputs(self,pid):
        cfg=self.config; my=self.hands[pid]; enm=self.hands[self.opponent(pid)]
        pad=lambda arr:(arr[:MAX_HAND]+[0]*MAX_HAND)[:MAX_HAND]
        return pad(my),pad(enm),\
               [self.scores[pid]/100.0, self.scores[self.opponent(pid)]/100.0,
                self.turn_count/10.0, len(my)/MAX_HAND, len(enm)/MAX_HAND,
                cfg['winScore']/200.0, cfg['maxTurns']/20.0, cfg['initHandCount']/10.0,
                self._res_ratio(pid), self._res_ratio(self.opponent(pid))]
    def best_attack(self,pid):
        enm=self.hands[self.opponent(pid)]; best,bg=None,0
        for i,a in enumerate(self.hands[pid]):
            if a==1: continue
            g=sum(n for n in enm if n%a==0)
            if g>bg: bg=g; best=(i,a,g)
        return best
    def best_add(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r=h[i]+h[j]
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def best_sub(self,pid):
        h=self.hands[pid]; best,bv=None,0
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r=h[i]-h[j]
                if r>0 and r>bv: bv=r; best=(i,j,r)
        return best
    def best_divprod(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j or h[j]==0: continue
                r=(h[i]//h[j])*(h[i]%h[j])
                if r>0 and r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def best_dsd(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i,num in enumerate(h):
            ds=digit_sum(num)
            if ds==0: continue
            q,r=divmod(num,ds)
            if q<=lim and q>bv: bv=q; best=(i,q,r)
        return best
    def best_gm(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                g=gcd(h[i],h[j])
                if g<=1: continue
                r=h[i]*g
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def get_optimal_action(self,pid):
        h=self.hands[pid]
        atk=self.best_attack(pid); add=self.best_add(pid); sub=self.best_sub(pid)
        dp=self.best_divprod(pid); dsd=self.best_dsd(pid); gm=self.best_gm(pid)
        ms=self.scores[pid]; win=self.config['winScore']
        def atk_ok(): return bool(atk) and self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=atk[0]])
        def sub_ok():
            if not sub: return False
            i,j,r=sub; return self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=i and k!=j]+[r])
        def dp_ok():
            if not dp: return False
            i,j,r=dp; return self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=i and k!=j]+[r])
        def dsd_ok():
            if not dsd: return False
            idx,q,r=dsd; nh=[c for k,c in enumerate(h) if k!=idx]+[q]+([] if r==0 else [r])
            return self._can_afford(pid,h,nh)
        if atk and self.turn_count>1 and ms+atk[2]>=win and atk_ok(): return 1
        if gm  and gm[2]>=30:  return 6
        if atk and self.turn_count>1 and atk[2]>=10 and atk_ok(): return 1
        if add and add[2]>=20: return 2
        if dp  and dp[2]>=15 and dp_ok(): return 4
        if atk and self.turn_count>1 and atk_ok(): return 1
        if gm  and gm[2]>=10: return 6
        if dsd and dsd[1]>=5 and dsd_ok(): return 5
        if add: return 2
        if dp and dp_ok(): return 4
        if sub and sub_ok(): return 3
        return 0
    def execute_action(self,pid,action):
        h=self.hands[pid]; eid=self.opponent(pid)
        if action==1:
            atk=self.best_attack(pid)
            if atk:
                i,v,gain=atk; b=h[:]; a=[c for k,c in enumerate(h) if k!=i]
                if self._can_afford(pid,b,a):
                    self._use_resource(pid,b,a); self.hands[eid]=[n for n in self.hands[eid] if n%v!=0]
                    h.pop(i); self.scores[pid]+=gain
                    if self.scores[pid]>=self.config['winScore']: self.done=True; self.winner=pid
        elif action==2:
            a=self.best_add(pid)
            if a:
                i,j,r=a; b=h[:]; h=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                self._use_resource(pid,b,h)
        elif action==3:
            s=self.best_sub(pid)
            if s:
                i,j,r=s; b=h[:]; nh=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==4:
            d=self.best_divprod(pid)
            if d:
                i,j,r=d; b=h[:]; nh=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==5:
            d=self.best_dsd(pid)
            if d:
                idx,q,r=d; b=h[:]; nh=[c for k,c in enumerate(h) if k!=idx]+[q]+([] if r==0 else [r])
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==6:
            g=self.best_gm(pid)
            if g:
                i,j,r=g; b=h[:]; h=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                self._use_resource(pid,b,h)
        self.hands[pid]=h; self.current_player=eid
        if self.current_player=='P1':
            self.turn_count+=1
            if self.turn_count>self.config['maxTurns']: self.done=True; self.winner=max(self.scores,key=lambda p:self.scores[p])
print('シミュレーター準備完了！')

In [ ]:
if os.path.exists(DATA_PATH):
    print('保存済みデータを読み込みます: ' + DATA_PATH)
    d=np.load(DATA_PATH)
    my_hands_data=d['my_hands']; enemy_hands_data=d['enemy_hands']
    scalars_data=d['scalars'];   y_data=d['y']
    print('読み込み完了！ %d 件' % len(y_data))
else:
    print('データ生成を開始します... (%d ゲーム × %d ルール)' % (NUM_EPISODES, len(RULE_CONFIGS)))
    my_hands_data,enemy_hands_data,scalars_data,y_data=[],[],[],[]
    for ep in range(NUM_EPISODES):
        cfg=random.choice(RULE_CONFIGS); game=MathCardGame(cfg); game.reset()
        while not game.done:
            pid=game.current_player
            my_h,enm_h,sc=game.get_inputs(pid)
            action=game.get_optimal_action(pid)
            my_hands_data.append(my_h); enemy_hands_data.append(enm_h)
            scalars_data.append(sc);    y_data.append(action)
            game.execute_action(pid,action)
        if (ep+1)%10000==0: print('%d/%d データ数: %d' % (ep+1,NUM_EPISODES,len(y_data)))
    my_hands_data=np.array(my_hands_data,dtype=np.int32)
    enemy_hands_data=np.array(enemy_hands_data,dtype=np.int32)
    scalars_data=np.array(scalars_data,dtype=np.float32)
    y_data=np.array(y_data,dtype=np.int32)
    print('生成完了！ 総データ数: %d' % len(y_data))
    np.savez_compressed(DATA_PATH,my_hands=my_hands_data,enemy_hands=enemy_hands_data,scalars=scalars_data,y=y_data)
    print('データを保存しました: ' + DATA_PATH)

In [ ]:
from tensorflow.keras.utils import to_categorical

idx=np.arange(len(y_data))
train_idx,val_idx=train_test_split(idx,test_size=0.1,random_state=42)

def make_dataset(indices, shuffle=True, bs=None):
    ds=tf.data.Dataset.from_tensor_slices((
        {'my_hand':my_hands_data[indices],'enemy_hand':enemy_hands_data[indices],'scalars':scalars_data[indices]},
        to_categorical(y_data[indices],NUM_ACTIONS)))
    if shuffle: ds=ds.shuffle(10000)
    return ds.batch(bs or BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset(train_idx); val_ds=make_dataset(val_idx,shuffle=False)

if os.path.exists(SAVE_PATH):
    print('保存済みモデルを読み込み中...')
    model=tf.keras.models.load_model(SAVE_PATH)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    print('読み込み完了！続きから学習します。')
else:
    print('新規モデルを作成します...')
    my_hand_in   =tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='my_hand')
    enemy_hand_in=tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='enemy_hand')
    scalar_in    =tf.keras.Input(shape=(SCALAR_DIM,),dtype='float32',name='scalars')
    math_emb   =tf.keras.layers.Embedding(MAX_CARD+1,MATH_DIM,weights=[FACTOR_TABLE],trainable=False,mask_zero=True,name='math_emb')
    learned_emb=tf.keras.layers.Embedding(MAX_CARD+1,EMBED_DIM,mask_zero=True,name='learned_emb')
    def encode_hand(hand_input):
        combined=tf.keras.layers.Concatenate(axis=-1)([math_emb(hand_input),learned_emb(hand_input)])
        return tf.keras.layers.GlobalMaxPooling1D()(combined)
    x=tf.keras.layers.Concatenate(name='combined')([encode_hand(my_hand_in),encode_hand(enemy_hand_in),scalar_in])
    x=tf.keras.layers.Dense(128,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.3)(x)
    x=tf.keras.layers.Dense(64, activation='relu')(x)
    x=tf.keras.layers.Dropout(0.2)(x)
    out=tf.keras.layers.Dense(NUM_ACTIONS,activation='softmax',name='action')(x)
    model=tf.keras.Model(inputs=[my_hand_in,enemy_hand_in,scalar_in],outputs=out)
    print('新規モデル作成完了！')

model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),loss='categorical_crossentropy',metrics=['accuracy'])
total=model.count_params(); trainable=sum(tf.size(w).numpy() for w in model.trainable_weights)
print('総パラメータ: %d  学習: %d  固定: %d' % (total,trainable,total-trainable))

## Optuna ハイパーパラメータ自動探索

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 新規特徴のインデックス（0〜8は固定、9〜13はOptuna選択対象）
_OPTIONAL_FEATS = {9:'素数フラグ', 10:'最小素因数', 11:'最大素因数', 12:'素因数個数Ω', 13:'約数の和σ'}

def _build_trial_model(trial):
    """Optuna trial 用モデルビルダー（特徴選択 + ハイパーパラメータ探索）"""
    # ── 特徴選択 ──
    mask = list(range(9))  # 0〜8は固定
    for idx, name in _OPTIONAL_FEATS.items():
        if trial.suggest_categorical(f'feat_{name}', [True, False]):
            mask.append(idx)
    ft       = ALL_FACTOR_TABLE[:, mask]
    math_dim = len(mask)

    # ── ハイパーパラメータ ──
    lr      = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    bs      = trial.suggest_categorical('batch_size', [128, 256, 512])
    units1  = trial.suggest_categorical('units1', [64, 128, 256])
    units2  = trial.suggest_categorical('units2', [32, 64, 128])
    drop1   = trial.suggest_float('dropout1', 0.1, 0.5)
    drop2   = trial.suggest_float('dropout2', 0.1, 0.4)
    emb_dim = trial.suggest_categorical('embed_dim', [8, 16, 32])

    my_in  = tf.keras.Input(shape=(MAX_HAND,),   dtype='int32',   name='my_hand')
    enm_in = tf.keras.Input(shape=(MAX_HAND,),   dtype='int32',   name='enemy_hand')
    sc_in  = tf.keras.Input(shape=(SCALAR_DIM,), dtype='float32', name='scalars')
    me = tf.keras.layers.Embedding(MAX_CARD+1, math_dim, weights=[ft],      trainable=False, mask_zero=True, name='math_emb')
    le = tf.keras.layers.Embedding(MAX_CARD+1, emb_dim,                     mask_zero=True,                  name='learned_emb')
    def enc(h): return tf.keras.layers.GlobalMaxPooling1D()(tf.keras.layers.Concatenate(axis=-1)([me(h), le(h)]))
    x = tf.keras.layers.Concatenate(name='combined')([enc(my_in), enc(enm_in), sc_in])
    x = tf.keras.layers.Dense(units1, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop1)(x)
    x = tf.keras.layers.Dense(units2, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop2)(x)
    out = tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax', name='action')(x)
    m = tf.keras.Model(inputs=[my_in, enm_in, sc_in], outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    return m, bs, mask

def _objective(trial):
    m, bs, _ = _build_trial_model(trial)
    tr = make_dataset(train_idx, bs=bs)
    va = make_dataset(val_idx,   bs=bs, shuffle=False)
    h = m.fit(tr, validation_data=va, epochs=OPTUNA_EPOCHS,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)],
              verbose=0)
    return max(h.history['val_accuracy'])

if USE_OPTUNA:
    print(f'Optuna 探索開始: {OPTUNA_TRIALS}試行 × {OPTUNA_EPOCHS}エポック')
    print(f'探索対象: 特徴選択(5種) + lr / batch / embed / units / dropout')
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best = study.best_params
    print(f'\n✅ 最適パラメータ (val_accuracy={study.best_value:.4f}):')

    # ── 選ばれた特徴を表示 ──
    best_mask = list(range(9))
    for idx, name in _OPTIONAL_FEATS.items():
        use = best.get(f'feat_{name}', False)
        mark = '✅' if use else '❌'
        print(f'  {mark} {name}')
        if use: best_mask.append(idx)
    print(f'  learning_rate: {best["learning_rate"]:.5f}')
    print(f'  batch_size:    {best["batch_size"]}')
    print(f'  embed_dim:     {best["embed_dim"]}')
    print(f'  units1/2:      {best["units1"]} / {best["units2"]}')

    # ── グローバル変数に反映 ──
    MATH_FEATURE_MASK = best_mask
    FACTOR_TABLE      = ALL_FACTOR_TABLE[:, MATH_FEATURE_MASK]
    MATH_DIM          = len(MATH_FEATURE_MASK)
    DIM_NAMES         = [ALL_DIM_NAMES[i] for i in MATH_FEATURE_MASK]
    LEARNING_RATE     = best['learning_rate']
    BATCH_SIZE        = best['batch_size']
    EMBED_DIM         = best['embed_dim']
    train_ds = make_dataset(train_idx, bs=BATCH_SIZE)
    val_ds   = make_dataset(val_idx,   bs=BATCH_SIZE, shuffle=False)
    print(f'\n→ 特徴次元: {MATH_DIM}  EMBED_DIM: {EMBED_DIM}')
    print('→「モデルの作成」セルを再実行してから学習してください！')
else:
    print('Optuna はスキップ（USE_OPTUNA=False）')

In [ ]:
class ImportanceTestCallback(tf.keras.callbacks.Callback):
    """IMPORTANCE_INTERVAL エポックごとに数学特徴の重要度テストを実行"""
    def on_epoch_end(self, epoch, logs=None):
        if IMPORTANCE_INTERVAL <= 0: return
        if (epoch + 1) % IMPORTANCE_INTERVAL != 0: return
        print(f'\n─── 重要度テスト (epoch {epoch+1}) ───')
        _, base_acc = self.model.evaluate(val_ds, verbose=0)
        for d in range(MATH_DIM):
            shuffled = FACTOR_TABLE.copy()
            shuffled[:,d] = np.random.permutation(shuffled[:,d])
            self.model.get_layer('math_emb').set_weights([shuffled])
            _, acc = self.model.evaluate(val_ds, verbose=0)
            drop = base_acc - acc
            self.model.get_layer('math_emb').set_weights([FACTOR_TABLE])
            bar = '█' * max(0, int(drop * 200)) + ('▒' if drop <= 0 else '')
            print(f'  {DIM_NAMES[d]:<12}: {drop*100:+.2f}%  {bar}')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=EARLY_STOP_PATIENCE, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=LR_REDUCE_PATIENCE),
    tf.keras.callbacks.ModelCheckpoint(filepath=SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ImportanceTestCallback(),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
best_val = max(history.history['val_accuracy'])
print('最高検証精度: %.1f%%' % (best_val * 100))

## 精度グラフ

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,key,title in zip(axes,['accuracy','loss'],['精度','損失']):
    ax.plot(history.history[key],label='訓練')
    ax.plot(history.history['val_'+key],label='検証')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

## 数学特徴の重要度

In [ ]:
import matplotlib.pyplot as plt
base_loss,base_acc=model.evaluate(val_ds,verbose=0)
print('ベースライン精度: %.2f%%' % (base_acc*100))
importances=[]
for d in range(MATH_DIM):
    shuffled=FACTOR_TABLE.copy()
    shuffled[:,d]=np.random.permutation(shuffled[:,d])
    model.get_layer('math_emb').set_weights([shuffled])
    _,acc=model.evaluate(val_ds,verbose=0)
    drop=base_acc-acc; importances.append(drop)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    print('次元%2d [%s]: %+.2f%%' % (d+1,DIM_NAMES[d],drop*100))
plt.figure(figsize=(10,4))
colors=['crimson' if v>0.02 else 'steelblue' for v in importances]
plt.barh(DIM_NAMES,[v*100 for v in importances],color=colors)
plt.axvline(0,color='black',linewidth=0.8)
plt.xlabel('精度の低下 (%)')
plt.title('数学特徴の重要度')
plt.tight_layout(); plt.show()
print('最も重要: ' + DIM_NAMES[int(np.argmax(importances))])

## TensorFlow.js エクスポート

In [ ]:
import shutil
OUT='tfjs_model'
tfjs.converters.save_keras_model(model,OUT)
print('エクスポート完了！')
for f in os.listdir(OUT): print('  %s (%.1f KB)' % (f,os.path.getsize(os.path.join(OUT,f))/1024))
shutil.make_archive('tfjs_model','zip','tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')

## 動作テスト

In [ ]:
names=['パス','攻撃','足し算','引き算','商x余','桁和分裂','GCD掛け']
std_cfg={'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,
         'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,'resourceLogBase':2}
tests=[
    {'p1':[6,3,10,7,2], 'p2':[12,18,5,9,4],    'desc':'攻撃チャンスあり'},
    {'p1':[5,8,3,2,7],  'p2':[1,1,1,1,1],       'desc':'合成が有利'},
    {'p1':[6,9,4,3,2],  'p2':[8,10,6,4,3],      'desc':'GCD有効'},
    {'p1':[36,12,24,48],'p2':[7,11,13,17,19,23],'desc':'相手が全員素数'}
]
for t in tests:
    g=MathCardGame(std_cfg); g.reset()
    g.hands['P1']=t['p1']; g.hands['P2']=t['p2']
    g.turn_count=3; g.scores={'P1':20,'P2':30}; g.resources={'P1':8.0,'P2':8.0}
    my_h,enm_h,sc=g.get_inputs('P1')
    pred=model.predict({'my_hand':np.array([my_h]),'enemy_hand':np.array([enm_h]),'scalars':np.array([sc])},verbose=0)[0]
    chosen=int(np.argmax(pred)); optimal=g.get_optimal_action('P1')
    mark='OK' if chosen==optimal else 'NG'
    print('[%s] %s  AI:%s  最適:%s' % (mark,t['desc'],names[chosen],names[optimal]))